In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from delta.tables import *

In [0]:
STRUCTURING_WINDOW_HOURS = 48
REPORTING_THRESHOLD_USD = 10000
NEAR_THRESHOLD_MIN_PCT = 0.85
NEAR_THRESHOLD_MAX_PCT = 0.99
MIN_DEPOSIT_COUNT = 3

transactions_df = spark.table("bankaml.silver.transactions")

deposits_df = transactions_df.filter(col("txn_type") == "deposit")

account_time_window = Window.partitionBy("account_id").orderBy("txn_ts")

deposits_df = deposits_df.withColumn(
    "prev_txn_ts", lag("txn_ts", 1).over(account_time_window)
)

deposits_df = deposits_df.withColumn(
    "gap_hours",
    when(
        col("prev_txn_ts").isNotNull(),
        (unix_timestamp("txn_ts") - unix_timestamp("prev_txn_ts")) / 3600.0
    ).otherwise(lit(None))
)

deposits_df = deposits_df.withColumn(
    "new_cluster_flag",
    when(
        col("prev_txn_ts").isNull() | (col("gap_hours") > STRUCTURING_WINDOW_HOURS),
        1
    ).otherwise(0)
)

deposits_df = deposits_df.withColumn(
    "cluster_id",
    sum("new_cluster_flag").over(account_time_window)
)

deposits_df = deposits_df.withColumn(
    "is_near_threshold",
    (col("amount_usd") >= REPORTING_THRESHOLD_USD * NEAR_THRESHOLD_MIN_PCT) &
    (col("amount_usd") <  REPORTING_THRESHOLD_USD * NEAR_THRESHOLD_MAX_PCT)
)

clustered_df = (deposits_df.groupBy("account_id", "cluster_id")
    .agg(
        count("txn_id").alias("deposit_count"),
        sum(when(col("is_near_threshold"), col("amount_usd")).otherwise(0)).alias("near_threshold_amount_usd"),
        sum("amount_usd").alias("total_amount_usd"),
        sum(col("is_near_threshold").cast("int")).alias("near_threshold_count"),
        collect_list("txn_id").alias("related_txn_ids"),
        min("txn_ts").alias("window_start"),
        max("txn_ts").alias("window_end"),
    )
)

structuring_flags_df = clustered_df.filter(
    col("near_threshold_count") >= MIN_DEPOSIT_COUNT
)

accounts_df = spark.table("bankaml.silver.accounts").filter(col("is_current") == "Y")

structuring_flags_df = (structuring_flags_df
    .join(accounts_df.select("account_id", "customer_id"), "account_id", "left")
    .withColumn("flag_id", sha2(concat_ws("_", "account_id", "cluster_id"), 256))
    .withColumn("flag_type", lit("structuring"))
    .withColumn("severity", lit("high"))
    .withColumn("flagged_ts", current_timestamp())
    .withColumn("status", lit("open"))
    .select(
        "flag_id", "account_id", "customer_id", "flag_type",
        "deposit_count", "near_threshold_count", "total_amount_usd",
        "related_txn_ids", "window_start", "window_end",
        "severity", "flagged_ts", "status"
    )
)

In [0]:
%sql
create table if not exists bankaml.gold.structuring_flags
(
    flag_id string,
    account_id string,
    customer_id string,
    flag_type string,
    deposit_count int,
    near_threshold_count int,
    related_txn_ids array<string>,
    window_start timestamp,
    window_end timestamp,
    severity string,
    flagged_ts timestamp,
    status string
)
using delta;

In [0]:
structuring_flags_table = DeltaTable.forName(spark, "bankaml.gold.structuring_flags")
structuring_flags_df = structuring_flags_df.select(*structuring_flags_table.toDF().columns)
(
    structuring_flags_table.alias("t").merge(
        structuring_flags_df.alias("s"), 
        "t.flag_id=s.flag_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)